# Workspace Setup (Admin-only)

This notebook provisions the shared resources for **GenAI Agent Evaluation on Databricks**. Run it once per workspace, as a workspace admin, before learners begin. It is idempotent: re-running it skips data that already exists, registers a new version of the shared agent, and redeploys the app.

**What it creates (all in `workspace.agent_eval`):**
1. `sf_airbnb_listings`: 50 San Francisco Airbnb listings from the **Airbnb Sample Data** Marketplace listing
2. Three Unity Gateway model services that learners use as a shared resource. The model each one routes to is set in `config/config-workspace-setup.yaml`:
    - `agent-model`: the model the agents call
    - `judge-model`: the model for the built-in and guideline judges
    - `custom-judge-model`: the model for the custom judges
3. The UC SQL function tools `avg_neigh_price` and `cnt_by_room_type` (DDL in `Includes/tools/sql/`)
4. The shared tool-calling agent (`Includes/shared_agent/`), registered in Unity Catalog as `workspace.agent_eval.airbnb` with the alias `champion`. Every demo loads this model
5. Read access on the data, and `EXECUTE` on the schema (the model services, the tools, and the registered agent) for all users
6. The shared **Airbnb assistant app**, `agent-airbnb-assistant` (from `agent-app/`), deployed on Databricks Apps. The app loads the same registered agent (`workspace.agent_eval.airbnb@champion`), so the app and the notebooks run the same agent. Its name starts with `agent`, so it also appears in the **Agents** menu. All users get `CAN_USE` on the app and `CAN_READ` on its MLflow experiment, and only the admin who runs this setup can manage it. The app's chat UI saves each user's thumbs up/down as human feedback on the trace, which learners use in the **Developer and SME Feedback with MLflow** lab
7. A scheduled **Repair-Resources** job (`Includes/Repair-Resources`) that recreates any shared resource that goes missing

Every step raises on failure, so the deployment stops at the first error and reports it.

**Alerting:** if a step fails, or the final check finds a resource missing, one message is posted to the Slack channel (`#training-automation-alert`) naming the failed step or the missing resources. Nothing is posted when setup succeeds. The Slack webhook URL is read from the Databricks secret `lab-alerts` / `slack-webhook` (see `_lib/slack_alert.py`); if the secret is missing, the alert is skipped with a warning and the run result is unchanged.

**Prerequisites:** the `dbacademy` catalog and the **Airbnb Sample Data** Marketplace listing (declared in `manifest.yaml`).

Run on **Serverless V6**.

## Install Libraries

In [0]:
import sys, os

includes_dir = "."
config_version = "workspace-setup"
sys.path.insert(0, os.path.abspath('.'))
config_path = f"{includes_dir}/config/config-{config_version}.yaml"
from _lib.library_installer import install_libraries
install_libraries(config_path)

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.abspath('.'))
config_path = "./config/config-workspace-setup.yaml"

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam

ws = WorkspaceClient()
admin_user = ws.current_user.me().user_name

# --- Slack alerting (see _lib/slack_alert.py) ---
from contextlib import contextmanager
from _lib.slack_alert import send_slack_alert

COURSE_NAME = "GenAI Agent Evaluation on Databricks"


@contextmanager
def setup_step(step):
    """Run one setup step. On failure, post a Slack alert naming the step, then re-raise so the run still fails."""
    try:
        yield
    except Exception as e:
        send_slack_alert(
            ws, dbutils,
            job_name="Workspace-Setup",
            summary=f"Workspace-Setup failed at step *{step}*. Shared lab resources may be incomplete.",
            course=COURSE_NAME,
            sections=[("Error", [f"{type(e).__name__}: {e}"])],
        )
        raise

## 1. Define Variables and Create the Shared Schema

In [0]:
SHARED_CATALOG = "workspace"  # this repo: course default "dbacademy" (Academy lab catalog)
SHARED_SCHEMA  = "agent_eval"
SHARED         = f"{SHARED_CATALOG}.{SHARED_SCHEMA}"

LISTINGS_TABLE = f"{SHARED}.sf_airbnb_listings"

# Airbnb Sample Data Marketplace share. Vocareum lab workspaces mount it under
# a dbacademy_ prefix (dbacademy_ + the dataset name in manifest.yaml).
AIRBNB_SHARE = "databricks_airbnb_sample_data"  # Marketplace share name outside Academy labs

# Unity Gateway model services and the models they route to, from
# config/config-workspace-setup.yaml (update models there, not here)
from _lib.config_loader import load_config

setup_cfg = load_config(config_path)
# This repo: the scheduled Repair-Resources job is optional (config: schedule_repair_job, default false)
SCHEDULE_REPAIR_JOB = bool(setup_cfg.get("schedule_repair_job", False))
MODEL_SERVICES = setup_cfg.get("model_services") or {}
RECREATE_ON_DRIFT = bool(setup_cfg.get("recreate_model_services_on_drift", False))
AGENT_MODEL_SERVICE = f"{SHARED}.agent-model"

# UC SQL function tools (DDL in Includes/tools/sql/) created in the shared schema
SHARED_TOOLS = ["avg_neigh_price", "cnt_by_room_type"]

# The shared agent, registered in Unity Catalog
AGENT_DIR            = Path(os.path.abspath("./shared_agent"))
AGENT_UC_MODEL       = f"{SHARED}.airbnb"
AGENT_ALIAS          = "champion"
REGISTRATION_EXPERIMENT = "/Shared/agent-eval-agent-registration"

# Shared agent app
APP_NAME        = "agent-airbnb-assistant"
APP_EXPERIMENT  = "/Shared/agent-eval-app"

# Group that learners belong to
LEARNER_GROUP = "users"

# Repair job (keep in sync with Repair-Resources)
REPAIR_JOB_NAME      = "Repair Resources"
REPAIR_NOTEBOOK_NAME = "Repair-Resources"
REPAIR_CRON          = "0 */10 * * * ?"   # every 10 minutes, UTC (TESTING ONLY: set back before merging)

with setup_step("Create schema"):
    # This repo: only create the catalog if it is missing. CREATE CATALOG IF NOT EXISTS fails on
    # metastores without a storage root even when the catalog already exists.
    if SHARED_CATALOG not in [r[0] for r in spark.sql("SHOW CATALOGS").collect()]:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {SHARED_CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SHARED}")
print(f"Using {SHARED}")

## 2. Load the Airbnb Listings

In [0]:
from _lib.process_data import process_csv_data

with setup_step("Load listings table"):
    if spark.catalog.tableExists(LISTINGS_TABLE):
        print(f"Table {LISTINGS_TABLE} already exists, skipping.")
    else:
        try:
            process_csv_data(AIRBNB_SHARE, LISTINGS_TABLE)
            print(f"Created {LISTINGS_TABLE} from /Volumes/{AIRBNB_SHARE}")
        except:
            process_csv_data("databricks_airbnb_sample_data", LISTINGS_TABLE)
            print(f"Created {LISTINGS_TABLE} from /Volumes/databricks_airbnb_sample_data")

## 3. Create the Model Services and the Shared Tools

In [0]:
from _lib.model_service_config import setup_model_service
from _lib.tool_manager import ToolManager

with setup_step("Create model services and tools"):
    model_service_fqns = {}
    for service_name, service in MODEL_SERVICES.items():
        served_by = service["served_by"]
        model_service_fqns[service_name] = setup_model_service(
            catalog_name=SHARED_CATALOG,
            schema_name=SHARED_SCHEMA,
            service_name=service_name,
            served_by=served_by,
            comment=f"{service.get('comment', 'Routes to ' + served_by)} (GenAI Agent Evaluation on Databricks)",
            recreate_on_drift=RECREATE_ON_DRIFT,
        )
        print(f"Model service: {model_service_fqns[service_name]} -> {served_by}")

    # ToolManager creates the functions in the current catalog and schema
    spark.sql(f"USE CATALOG {SHARED_CATALOG}")
    spark.sql(f"USE SCHEMA {SHARED_SCHEMA}")
    tools_dir = os.path.abspath("./tools/sql")
    ToolManager(catalog_name=SHARED_CATALOG, schema_name=SHARED_SCHEMA).create_tools(
        {(tool, tools_dir) for tool in SHARED_TOOLS}
    )

## 4. Register the Shared Agent in Unity Catalog

Renders the agent's config with the shared model service and tools, logs `Includes/shared_agent/airbnb_agent.py` as an MLflow model, registers it as `workspace.agent_eval.airbnb`, and moves the `champion` alias to the new version. The demos and the app load the model by this alias.

In [0]:
import mlflow
from mlflow import MlflowClient
from mlflow.models.resources import DatabricksFunction
from _lib.config_renderer import ConfigRenderer

with setup_step("Register shared agent"):
    rendered_config = Path(os.path.abspath("./artifacts/shared_agent/airbnb_agent_config.yaml"))
    renderer = ConfigRenderer(artifacts_dir=rendered_config.parent)
    renderer.update_yaml_config(
        renderer.render_text_template(
            template_path=AGENT_DIR / "airbnb_agent_config.yaml",
            substitutions={
                "LLM_ENDPOINT_NAME": AGENT_MODEL_SERVICE,
                "CATALOG_NAME": SHARED_CATALOG,
                "SCHEMA_NAME": SHARED_SCHEMA,
                "TOOL1": SHARED_TOOLS[0],
                "TOOL2": SHARED_TOOLS[1],
            },
        ),
        rendered_config,
    )

    pip_requirements = [
        line.strip()
        for line in (AGENT_DIR / "requirements.txt").read_text().splitlines()
        if line.strip() and not line.startswith("#")
    ]

    mlflow.set_tracking_uri("databricks")
    mlflow.set_registry_uri("databricks-uc")
    mlflow.set_experiment(REGISTRATION_EXPERIMENT)

    with mlflow.start_run(run_name="register-airbnb-agent", tags={
        "training_type": "db_academytraining",
        "model": AGENT_MODEL_SERVICE,
        "agent_type": "TOOL-CALLING",
        "agent_id": "airbnb",
    }):
        logged = mlflow.pyfunc.log_model(
            name="airbnb",
            python_model=str(AGENT_DIR / "airbnb_agent.py"),
            model_config=str(rendered_config),
            input_example={"input": [{"role": "user", "content": "What is the average listing price in Mission?"}]},
            pip_requirements=pip_requirements,
            resources=[DatabricksFunction(function_name=f"{SHARED}.{tool}") for tool in SHARED_TOOLS],
            registered_model_name=AGENT_UC_MODEL,
        )

    agent_version = logged.registered_model_version
    MlflowClient().set_registered_model_alias(AGENT_UC_MODEL, AGENT_ALIAS, agent_version)
    print(f"Registered {AGENT_UC_MODEL} version {agent_version} with alias '{AGENT_ALIAS}'")

## 5. Grant Access to All Users

Learners read the listings, call the model services and tools, and load the registered agent. `EXECUTE` on the schema covers all three: model services, functions, and registered models.

In [0]:
with setup_step("Grant permissions"):
    spark.sql(f"GRANT USE CATALOG ON CATALOG {SHARED_CATALOG} TO `account users`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {SHARED} TO `account users`")
    spark.sql(f"GRANT SELECT ON TABLE {LISTINGS_TABLE} TO `account users`")
    spark.sql(f"GRANT EXECUTE ON SCHEMA {SHARED} TO `account users`")
    print(f"Granted SELECT on {LISTINGS_TABLE} and EXECUTE on {SHARED} to account users")

## 6. Deploy the Shared Agent App

Deploys `agent-app/` (next to this `Includes/` folder) as a Databricks App. The app loads `workspace.agent_eval.airbnb@champion` from Unity Catalog at startup. Its service principal gets `EXECUTE` on the `workspace.agent_eval` schema (the registered agent, the model services, and the tools) and `SELECT` on the listings the tools query. All users get `CAN_USE` on the app and `CAN_READ` on its experiment.

After you register a new version of the agent (section 4), redeploy or restart the app so it loads the new `champion` version.

In [0]:
from databricks.sdk.errors import NotFound
from databricks.sdk.service.workspace import ObjectType
from _lib.app_deployer import AppDeployer
from _lib.create_files_folders import create_file

with setup_step("Deploy agent app"):
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    course_dir = os.path.dirname(os.path.dirname(notebook_path))  # .../<course>/Includes/Workspace-Setup -> .../<course>
    agent_app_path = f"{course_dir}/agent-app"

    # Importing the course into the workspace can turn agent_server/__init__.py into
    # a notebook named __init__, which breaks the app's package build. Replace it
    # with a real Python file before deploying.
    init_notebook = f"{agent_app_path}/agent_server/__init__"
    try:
        if ws.workspace.get_status(init_notebook).object_type == ObjectType.NOTEBOOK:
            ws.workspace.delete(init_notebook)
            print(f"Removed notebook {init_notebook}")
    except NotFound:
        pass
    fs_app_path = agent_app_path if agent_app_path.startswith("/Workspace") else f"/Workspace{agent_app_path}"
    create_file(f"{fs_app_path}/agent_server/__init__.py", '"""Airbnb assistant app package."""\n')

    app_result = AppDeployer(
        catalog_name=SHARED_CATALOG,
        schema_name=SHARED_SCHEMA,
        agent_app_path=agent_app_path,
        username=admin_user,
        workspace_client=ws,
        app_name=APP_NAME,
        experiment_path=APP_EXPERIMENT,
        app_description="Airbnb assistant for GenAI Agent Evaluation on Databricks",
        # EXECUTE on the schema covers the registered agent, the model services, and the tools
        grant_model_service_execute=True,
        extra_grants=[("TABLE", LISTINGS_TABLE, "SELECT")],
        share_with_groups=[LEARNER_GROUP],
        extra_env_vars={
            "AGENT_MODEL": AGENT_UC_MODEL,
            "AGENT_ALIAS": AGENT_ALIAS,
            "MODEL_SERVICE": AGENT_MODEL_SERVICE,
        },
    ).deploy(smoke_test=True)

    # Learners may use the app but must not manage or delete it. Replace (not add to)
    # the app's permission list, so no broader grant from an earlier run or manual
    # change survives: the admin running setup manages it, learners can only use it.
    # Workspace admins always have CAN_MANAGE, so learners must not be admins.
    ws.permissions.set(
        request_object_type="apps",
        request_object_id=APP_NAME,
        access_control_list=[
            iam.AccessControlRequest(user_name=admin_user, permission_level=iam.PermissionLevel.CAN_MANAGE),
            iam.AccessControlRequest(group_name=LEARNER_GROUP, permission_level=iam.PermissionLevel.CAN_USE),
        ],
    )
    print(f"App '{APP_NAME}' permissions:")
    for entry in ws.permissions.get(request_object_type="apps", request_object_id=APP_NAME).access_control_list or []:
        who = entry.group_name or entry.user_name or entry.service_principal_name
        levels = ", ".join(p.permission_level.value for p in entry.all_permissions or [])
        print(f"  {who}: {levels}")

## 7. Deploy the Repair-Resources Job

Creates (or updates) a scheduled job that runs `Includes/Repair-Resources`. It checks every shared resource, recreates only what is missing, and fails if a repair does not work.

In [0]:
from databricks.sdk.service.jobs import CronSchedule, JobSettings, NotebookTask, PauseStatus, Task

if not SCHEDULE_REPAIR_JOB:
    print("Skipping the scheduled Repair-Resources job (schedule_repair_job: false in config-workspace-setup.yaml)")
else:
    with setup_step("Deploy Repair Resources job"):
        REPAIR_PATH = f"{os.path.dirname(notebook_path)}/{REPAIR_NOTEBOOK_NAME}"

        job_settings = JobSettings(
            name=REPAIR_JOB_NAME,
            tasks=[Task(task_key="repair_resources", notebook_task=NotebookTask(notebook_path=REPAIR_PATH))],
            schedule=CronSchedule(quartz_cron_expression=REPAIR_CRON, timezone_id="UTC", pause_status=PauseStatus.UNPAUSED),
        )

        existing_jobs = list(ws.jobs.list(name=REPAIR_JOB_NAME))
        if existing_jobs:
            job_id = existing_jobs[0].job_id
            ws.jobs.reset(job_id=job_id, new_settings=job_settings)
            print(f"Updated existing job '{REPAIR_JOB_NAME}' (job_id={job_id})")
        else:
            job_id = ws.jobs.create(name=job_settings.name, tasks=job_settings.tasks, schedule=job_settings.schedule).job_id
            print(f"Created job '{REPAIR_JOB_NAME}' (job_id={job_id})")

        print(f"  Schedule: {REPAIR_CRON} (UTC)")
        print(f"  Notebook: {REPAIR_PATH}")

## 8. Verify Resources

Re-checks every resource. If anything is missing, posts a Slack alert listing it and raises, so the deployment is reported as failed.

In [0]:
from databricks.sdk.service.apps import ApplicationState
from _lib.model_service_config import _get_service

missing = []


def _check(name, fn):
    try:
        ok, details = bool(fn()), ""
    except Exception as e:
        ok, details = False, f"{type(e).__name__}: {str(e)[:300]}"
    print(f"  [{'OK' if ok else 'MISSING'}] {name}" + (f" -> {details}" if details else ""))
    if not ok:
        missing.append(f"{name}: {details or 'not found'}")


def _has_grant(sec_type, sec_name, privilege):
    rows = spark.sql(f"SHOW GRANTS ON {sec_type} {sec_name}").collect()
    return any(r["Principal"] == "account users" and r["ActionType"] == privilege for r in rows)


def _app_running():
    status = ws.apps.get(APP_NAME).app_status
    return bool(status and status.state == ApplicationState.RUNNING)


def _app_can_use():
    perms = ws.permissions.get(request_object_type="apps", request_object_id=APP_NAME)
    allowed = {iam.PermissionLevel.CAN_USE, iam.PermissionLevel.CAN_MANAGE}
    return any(
        acl.group_name == LEARNER_GROUP and any(p.permission_level in allowed for p in (acl.all_permissions or []))
        for acl in (perms.access_control_list or [])
    )


_check(f"Catalog {SHARED_CATALOG}", lambda: ws.catalogs.get(SHARED_CATALOG))
_check(f"Schema {SHARED}", lambda: ws.schemas.get(SHARED))
_check(f"Table {LISTINGS_TABLE} (has rows)", lambda: spark.table(LISTINGS_TABLE).limit(1).count() > 0)
for service_name in MODEL_SERVICES:
    _check(f"Model service {SHARED}.{service_name}", lambda n=service_name: _get_service(ws, f"{SHARED}.{n}"))
for tool in SHARED_TOOLS:
    _check(f"Function {SHARED}.{tool}", lambda t=tool: ws.functions.get(f"{SHARED}.{t}"))
_check(f"Agent {AGENT_UC_MODEL}@{AGENT_ALIAS}", lambda: ws.model_versions.get_by_alias(AGENT_UC_MODEL, AGENT_ALIAS))

for sec_type, sec_name, priv in [
    ("CATALOG", SHARED_CATALOG, "USE CATALOG"),
    ("SCHEMA", SHARED, "USE SCHEMA"),
    ("TABLE", LISTINGS_TABLE, "SELECT"),
    ("SCHEMA", SHARED, "EXECUTE"),
]:
    _check(f"{priv} ON {sec_type} {sec_name} -> account users",
           lambda t=sec_type, n=sec_name, p=priv: _has_grant(t, n, p))

_check(f"App {APP_NAME} RUNNING", _app_running)
_check(f"App {APP_NAME} CAN_USE -> {LEARNER_GROUP}", _app_can_use)
if SCHEDULE_REPAIR_JOB:
    _check(f"Job {REPAIR_JOB_NAME}", lambda: list(ws.jobs.list(name=REPAIR_JOB_NAME)))

if missing:
    send_slack_alert(
        ws, dbutils,
        job_name="Workspace-Setup",
        summary=f"Workspace-Setup finished but *{len(missing)}* shared resource(s) are missing.",
        course=COURSE_NAME,
        sections=[("Missing resources", missing)],
    )
    raise RuntimeError(
        f"Workspace-Setup verification failed: {len(missing)} resource(s) missing:\n"
        + "\n".join(f"  - {m}" for m in missing)
    )
print("All resources present.")

## Summary

In [0]:
print("=" * 60)
print("Workspace Setup Complete")
print("=" * 60)
print(f"  Shared table      : {LISTINGS_TABLE}")
for service_name, fqn in model_service_fqns.items():
    print(f"  Model service     : {fqn} -> {MODEL_SERVICES[service_name]['served_by']}")
print(f"  Shared tools      : {', '.join(f'{SHARED}.{t}' for t in SHARED_TOOLS)}")
print(f"  Registered agent  : {AGENT_UC_MODEL} version {agent_version} (@{AGENT_ALIAS})")
print(f"  Agent app         : {app_result['app_name']} ({app_result['app_status']})")
print(f"  App URL           : {app_result['app_url']}")
print(f"  App experiment    : {APP_EXPERIMENT}")
print(f"  Smoke test passed : {app_result['smoke_test_passed']}")
print(f"  Repair job        : {REPAIR_JOB_NAME} ({REPAIR_CRON} UTC)")